# Import Library

In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
import pillow_heif
import seaborn as sns
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras import mixed_precision

# Load Dataset

In [2]:
day = r"D:\KULIAH\SEMESTER 5\DEEP LEARNING\PROJECT\DATASET\DATA SEKUNDER\Day"
night = r"D:\KULIAH\SEMESTER 5\DEEP LEARNING\PROJECT\DATASET\DATA SEKUNDER\Night"

day_Resized = r"D:\KULIAH\SEMESTER 5\DEEP LEARNING\PROJECT\DATASET\DATA SEKUNDER\Resized\Day"
night_Resized = r"D:\KULIAH\SEMESTER 5\DEEP LEARNING\PROJECT\DATASET\DATA SEKUNDER\Resized\Night"

os.makedirs(day_Resized, exist_ok=True)
os.makedirs(night_Resized, exist_ok=True)

In [3]:
mixed_precision.set_global_policy('float32')
print("Precision Policy:", mixed_precision.global_policy())

Precision Policy: <Policy "float32">


## Image Resize

In [ ]:
def resize_images(input_dir, output_dir, size=(256, 256)):
    for f in os.listdir(input_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            path = os.path.join(input_dir, f)
            try:
                img = Image.open(path).convert("RGB")
                img = img.resize(size)
                img.save(os.path.join(output_dir, f))
            except Exception as e:
                print(f"Gagal resize {f}: {e}")

resize_images(day, day_Resized)
resize_images(night, night_Resized)

In [4]:
AUTOTUNE = tf.data.AUTOTUNE
strategy = tf.distribute.MirroredStrategy()
print("GPU Devices:", strategy.num_replicas_in_sync)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
GPU Devices: 1


In [6]:
day_images = tf.io.gfile.glob(os.path.join(day_Resized, "*.jpg"))
night_images = tf.io.gfile.glob(os.path.join(night_Resized, "*.jpg"))
print(f"Total Day: {len(day_images)}, Night: {len(night_images)}")

Total Day: 9000, Night: 6000


In [7]:
def preprocess_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0
    return image

BATCH_SIZE = 4
day_ds = tf.data.Dataset.from_tensor_slices(day_images).map(preprocess_image, num_parallel_calls=AUTOTUNE)
night_ds = tf.data.Dataset.from_tensor_slices(night_images).map(preprocess_image, num_parallel_calls=AUTOTUNE)
day_ds = day_ds.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
night_ds = night_ds.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [8]:
class InstanceNormalization(layers.Layer):
    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.scale = self.add_weight(name="scale", shape=(channels,), initializer="ones", trainable=True, dtype=tf.float32)
        self.offset = self.add_weight(name="offset", shape=(channels,), initializer="zeros", trainable=True, dtype=tf.float32)

    def call(self, x):
        x32 = tf.cast(x, tf.float32)
        mean, var = tf.nn.moments(x32, axes=[1, 2], keepdims=True)
        inv = tf.math.rsqrt(var + tf.cast(self.epsilon, tf.float32))
        normalized = (x32 - mean) * inv
        out32 = self.scale * normalized + self.offset
        return tf.cast(out32, x.dtype)

In [9]:
def Generator(input_shape=(256, 256, 3), ngf=64, n_down=4):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    skips = []

    # Encoder
    for i in range(n_down):
        filters = ngf * (2 ** i)
        if i != 0:
            skips.append(x)
        x = layers.Conv2D(filters, 4, strides=2, padding='same', use_bias=False,
                          kernel_initializer='he_normal')(x)
        if i != 0:
            x = InstanceNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)

    # Bottleneck
    bottleneck_filters = ngf * (2 ** n_down)
    x = layers.Conv2D(bottleneck_filters, 4, strides=1, padding='same',
                      use_bias=False, kernel_initializer='he_normal')(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)

    # Decoder
    for i in reversed(range(n_down)):
        filters = ngf * (2 ** i)
        x = layers.Conv2DTranspose(filters, 4, strides=2, padding='same',
                                   use_bias=False, kernel_initializer='he_normal')(x)
        x = InstanceNormalization()(x)
        if i < 2:
            x = layers.Dropout(0.5)(x)
        x = layers.ReLU()(x)
        if len(skips) > 0:
            skip = skips.pop()
            if skip.shape[1] == x.shape[1] and skip.shape[2] == x.shape[2]:
                x = layers.Concatenate()([x, skip])

    out = layers.Conv2DTranspose(3, 4, strides=1, padding='same',
                                 activation='tanh', kernel_initializer='glorot_uniform')(x)
    out = tf.cast(out, tf.float32)
    return Model(inputs=inputs, outputs=out, name="U_Net_Generator")

In [10]:
def Discriminator(input_shape=(256, 256, 3), ndf=64, n_layers=3):
    inp = layers.Input(shape=input_shape)
    x = inp
    x = layers.Conv2D(ndf, 4, strides=2, padding='same', kernel_initializer='he_normal')(x)
    x = layers.LeakyReLU(0.2)(x)

    nf_mult = 1
    for n in range(1, n_layers):
        nf_mult_prev = nf_mult
        nf_mult = min(2 ** n, 8)
        x = layers.Conv2D(ndf * nf_mult, 4,
                          strides=2 if n < n_layers - 1 else 1,
                          padding='same', use_bias=False,
                          kernel_initializer='he_normal')(x)
        x = InstanceNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)

    out = layers.Conv2D(1, 4, strides=1, padding='same',
                        kernel_initializer='glorot_uniform')(x)
    return Model(inputs=inp, outputs=out, name="PatchGAN_Discriminator")

In [11]:
mse = tf.keras.losses.MeanSquaredError()

def discriminator_loss(real, generated):
    real_loss = mse(tf.ones_like(real), real)
    gen_loss = mse(tf.zeros_like(generated), generated)
    return 0.5 * (real_loss + gen_loss)

def generator_loss(generated):
    return mse(tf.ones_like(generated), generated)

def calc_cycle_loss(real_image, cycled_image, LAMBDA=10):
    real_image = tf.cast(real_image, tf.float32)
    cycled_image = tf.cast(cycled_image, tf.float32)
    return LAMBDA * tf.reduce_mean(tf.abs(real_image - cycled_image))

def identity_loss(real_image, same_image, LAMBDA=5):
    real_image = tf.cast(real_image, tf.float32)
    same_image = tf.cast(same_image, tf.float32)
    return LAMBDA * 0.5 * tf.reduce_mean(tf.abs(real_image - same_image))

In [15]:
initial_lr = 2e-4
steps_per_epoch = max(1, min(len(day_images), len(night_images)) // BATCH_SIZE)
lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[50 * steps_per_epoch, 75 * steps_per_epoch],
    values=[initial_lr, initial_lr * 0.1, initial_lr * 0.01]
)

with strategy.scope():
    day2night_gen = Generator()
    night2day_gen = Generator()
    day_disc = Discriminator()
    night_disc = Discriminator()

    gen_opt = tf.keras.optimizers.Adam(lr_schedule, beta_1=0.5)
    disc_opt = tf.keras.optimizers.Adam(lr_schedule, beta_1=0.5)

    # --- Fix KeyError: register all model variables to optimizer ---
    gen_opt.build(day2night_gen.trainable_variables + night2day_gen.trainable_variables)
    disc_opt.build(day_disc.trainable_variables + night_disc.trainable_variables)

In [16]:
@tf.function
def train_step(real_day, real_night):
    with tf.GradientTape(persistent=True) as tape:
        fake_night = day2night_gen(real_day, training=True)
        cycled_day = night2day_gen(fake_night, training=True)

        fake_day = night2day_gen(real_night, training=True)
        cycled_night = day2night_gen(fake_day, training=True)

        same_day = night2day_gen(real_day, training=True)
        same_night = day2night_gen(real_night, training=True)

        D_real_day = day_disc(real_day, training=True)
        D_fake_day = day_disc(fake_day, training=True)
        D_real_night = night_disc(real_night, training=True)
        D_fake_night = night_disc(fake_night, training=True)

        g_day_loss = generator_loss(D_fake_day)
        g_night_loss = generator_loss(D_fake_night)
        cyc_loss = calc_cycle_loss(real_day, cycled_day) + calc_cycle_loss(real_night, cycled_night)
        id_loss = identity_loss(real_day, same_day) + identity_loss(real_night, same_night)

        total_day2night_loss = g_night_loss + cyc_loss + id_loss
        total_night2day_loss = g_day_loss + cyc_loss + id_loss

        day_disc_loss = discriminator_loss(D_real_day, D_fake_day)
        night_disc_loss = discriminator_loss(D_real_night, D_fake_night)

    gen_grads = tape.gradient(total_day2night_loss, day2night_gen.trainable_variables)
    night_grads = tape.gradient(total_night2day_loss, night2day_gen.trainable_variables)
    disc_day_grads = tape.gradient(day_disc_loss, day_disc.trainable_variables)
    disc_night_grads = tape.gradient(night_disc_loss, night_disc.trainable_variables)

    gen_opt.apply_gradients(zip(gen_grads, day2night_gen.trainable_variables))
    gen_opt.apply_gradients(zip(night_grads, night2day_gen.trainable_variables))
    disc_opt.apply_gradients(zip(disc_day_grads, day_disc.trainable_variables))
    disc_opt.apply_gradients(zip(disc_night_grads, night_disc.trainable_variables))

    return total_day2night_loss, total_night2day_loss, day_disc_loss, night_disc_loss

In [ ]:
EPOCHS = 50
history = {'day2night_gen': [], 'night2day_gen': [], 'day_disc': [], 'night_disc': []}

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    d2n_gen, n2d_gen, d_disc, n_disc, n_batches = 0, 0, 0, 0, 0

    for real_day, real_night in tf.data.Dataset.zip((day_ds, night_ds)):
        d2n_gen_loss, n2d_gen_loss, d_disc_loss, n_disc_loss = train_step(real_day, real_night)
        d2n_gen += d2n_gen_loss.numpy()
        n2d_gen += n2d_gen_loss.numpy()
        d_disc += d_disc_loss.numpy()
        n_disc += n_disc_loss.numpy()
        n_batches += 1

    history['day2night_gen'].append(d2n_gen/n_batches)
    history['night2day_gen'].append(n2d_gen/n_batches)
    history['day_disc'].append(d_disc/n_batches)
    history['night_disc'].append(n_disc/n_batches)

    print(f"Day→Night Gen: {d2n_gen/n_batches:.4f}, Night→Day Gen: {n2d_gen/n_batches:.4f}, "
          f"Day Disc: {d_disc/n_batches:.4f}, Night Disc: {n_disc/n_batches:.4f}")


Epoch 1/50


In [ ]:
# history = {'day2night_gen': [], 'night2day_gen': [], 'day_disc': [], 'night_disc': []}

# EPOCHS = 50
# for epoch in range(EPOCHS):
#     print(f"\nEpoch {epoch+1}/{EPOCHS}")
#     d2n_gen, n2d_gen, d_disc, n_disc, n_batches = 0, 0, 0, 0, 0

#     for real_day, real_night in tf.data.Dataset.zip((day_ds, night_ds)):
#         d2n_gen_loss, n2d_gen_loss, d_disc_loss, n_disc_loss = train_step(real_day, real_night)
#         d2n_gen += d2n_gen_loss.numpy()
#         n2d_gen += n2d_gen_loss.numpy()
#         d_disc += d_disc_loss.numpy()
#         n_disc += n_disc_loss.numpy()
#         n_batches += 1

#     history['day2night_gen'].append(d2n_gen/n_batches)
#     history['night2day_gen'].append(n2d_gen/n_batches)
#     history['day_disc'].append(d_disc/n_batches)
#     history['night_disc'].append(n_disc/n_batches)

#     print(f"Day→Night Gen: {d2n_gen/n_batches:.4f}, Night→Day Gen: {n2d_gen/n_batches:.4f}, "
#           f"Day Disc: {d_disc/n_batches:.4f}, Night Disc: {n_disc/n_batches:.4f}")